### Cell 1: Setup and Imports

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from PIL import Image
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Cell 2: Dataset Paths & Basic EDA

In [ ]:
data_dir = 'chest_xray'

# Check class distribution
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        path = os.path.join(data_dir, split, cls)
        if os.path.exists(path):
            print(f'{split} - {cls}: {len(os.listdir(path))} images')

# Plot sample image
sample_path = os.path.join(data_dir, 'train', 'NORMAL', os.listdir(os.path.join(data_dir, 'train', 'NORMAL'))[0])
plt.imshow(Image.open(sample_path), cmap='gray')
plt.title('Sample Normal X-Ray')
plt.show()

### Cell 3: Data Transforms & Loaders

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
val_data = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)
test_data = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

print("DataLoaders created.")

### Cell 4: Define Model (ResNet18)

In [ ]:
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Binary classification
model = model.to(device)
print("Model loaded.")

### Cell 5: Loss, Optimizer, Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
print("Loss, optimizer, scheduler configured.")

### Cell 6: Training Function

In [ ]:
def train_model(model, loader, criterion, optimizer, epochs=5):
    history = {'loss': [], 'acc': []}
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_corrects = 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(torch.max(outputs, 1)[1] == labels)
        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc = running_corrects.double() / len(loader.dataset)
        history['loss'].append(epoch_loss)
        history['acc'].append(epoch_acc.item())
        print(f'Epoch {epoch+1}/{epochs} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
    return history

### Cell 7: Run Training

In [ ]:
history = train_model(model, train_loader, criterion, optimizer, epochs=5)

### Cell 8: Loss/Accuracy Plots

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['loss'], label='Loss')
plt.legend(); plt.title('Loss')
plt.subplot(1, 2, 2)
plt.plot(history['acc'], label='Accuracy')
plt.legend(); plt.title('Accuracy')
plt.show()

### Cell 9: Test Evaluation

In [ ]:
def evaluate(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = torch.max(outputs, 1)[1]
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
    return y_true, y_pred

y_true, y_pred = evaluate(model, test_loader)

### Cell 10: Confusion Matrix & Classification Report

In [ ]:
print(classification_report(y_true, y_pred, target_names=test_data.classes))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=test_data.classes, yticklabels=test_data.classes)
plt.ylabel('Actual'); plt.xlabel('Predicted'); plt.show()

### Cell 11: Identify Target Layer for Grad-CAM

In [ ]:
# For ResNet18, the last convolutional layer is 'layer4'
target_layer = model.layer4
print(f"Target layer for Grad-CAM: {target_layer}")

### Cell 12: Save Model and Download

In [ ]:
# Save model weights
torch.save(model.state_dict(), 'model.pth')
print("Model saved as model.pth")

# Download instructions for Google Colab
try:
    from google.colab import files
    files.download('model.pth')
except ImportError:
    print("Not running in Colab, file saved locally as model.pth")

### Cell 13: Grad-CAM Implementation

In [ ]:
import torch.nn.functional as F

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Hook to capture activations
        self.target_layer.register_forward_hook(self.save_activation)
        # Hook to capture gradients
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_image, target_class=None):
        self.model.eval()
        output = self.model(input_image)
        if target_class is None:
            target_class = torch.argmax(output)
        
        self.model.zero_grad()
        output[0, target_class].backward()
        
        # Global Average Pooling of gradients
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        
        # Weight activations by gradients
        activations = self.activations.detach()
        for i in range(activations.shape[1]):
            activations[:, i, :, :] *= pooled_gradients[i]
        
        # Heatmap
        heatmap = torch.mean(activations, dim=1).squeeze()
        heatmap = F.relu(heatmap)
        heatmap /= torch.max(heatmap)
        return heatmap.cpu().detach().numpy()

### Cell 14: Inference Function

In [ ]:
def inference(image_path, model, target_layer):
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    img = Image.open(image_path).convert('RGB')
    input_tensor = preprocess(img).unsqueeze(0).to(device)
    
    model.eval()
    output = model(input_tensor)
    pred_label = torch.argmax(output).item()
    conf = F.softmax(output, dim=1)[0][pred_label].item()
    
    cam = GradCAM(model, target_layer)
    heatmap = cam.generate(input_tensor, target_class=pred_label)
    
    return pred_label, conf, heatmap

### Cell 15: Test Inference

In [ ]:
# Assuming 'model' and 'target_layer' are already defined from training cells
sample_image = 'test_sample.jpg'  # Replace with actual path
if os.path.exists(sample_image):
    label, confidence, heatmap = inference(sample_image, model, target_layer)

    original_img = cv2.imread(sample_image)
    original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    heatmap_resized = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
    heatmap_resized = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)

    overlay = cv2.addWeighted(original_img, 0.6, heatmap_colored, 0.4, 0)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(original_img)
    plt.title('Original')
    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title(f'Label: {label}, Conf: {confidence:.2f}')
    plt.show()
else:
    print(f"File {sample_image} not found. Please provide a valid path.")